# ARTICLE Pipeline — Notebook 3: Heston + SBTS Training

**Folder:** `ARTICLE_SBTS` | **Session:** Heston, SBTS | **Runs:** 120 | **Est. runtime:** ~20h on Colab T4

## Q1 journal reproducibility — what this notebook guarantees

Each training run saves a full 5-layer checkpoint:

1. **Model state** — `net_state_dict`, `V0`, `optimizer_state_dict`,
   `scheduler_state_dict`, CVaR `nu`
2. **Architecture + hyperparameters** — complete spec to rebuild the
   network and reproduce training end-to-end
3. **Training history** — per-epoch train/val loss, learning rate,
   gradient norms (pre/post clip), `V0`, `nu`
4. **Evaluation artifacts** — FULL per-path arrays (residuals, PnL,
   cost, payoff) on synthetic test + 4 historical periods, with
   SHA-256 hashes
5. **Provenance** — timestamps, environment snapshot (torch / CUDA /
   GPU), data-file SHA-256, RNG state, Git commit, integrity hash

For each run, we write:

| File | Purpose |
|------|---------|
| `{key}_mse.pt` | Full Phase 1 checkpoint |
| `{key}_mse.json` | Sidecar metadata (fast inspection) |
| `{key}_cvar.pt` | Full Phase 2 (curriculum) checkpoint |
| `{key}_cvar.json` | Sidecar metadata |

End-of-session: `manifest_article_{SESSION}.json` aggregates all
checkpoint hashes for top-level audit.

## Curriculum schedule (unchanged from thesis)

- **Phase 1 (MSE)**: 500 epochs max, LR 1e-3, patience 20. `V0` trainable.
- **Phase 2 (CVaR curriculum)**: Load best MSE weights, freeze `V0`,
  fine-tune with `CVaR_95` loss at LR 1e-4 for 200 epochs, patience 20.

## Inputs required (from `article_1_data.ipynb`)

- `{ds}_deep_hedging.npz` (one per generator in this session)
- `historical_test_paths.npz` (4 periods: COVID_2020, PostCOVID_2021_22,
  Normal_2023_24, Recent_2025)


In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 1: SETUP + DATA LOADING
# ═════════════════════════════════════════════════════════════════

import os, sys, time, gc, json, io, hashlib, datetime, platform, subprocess, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Device ────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  Memory: {props.total_memory/1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ═════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ═════════════════════════════════════════════════════════════════

DRIVE_FOLDER = '/content/drive/MyDrive/ARTICLE_SBTS'
SESSION_NAME = 'Heston_SBTS'
DS_LIST      = ['Heston', 'SBTS']

CKPT_ROOT    = os.path.join(DRIVE_FOLDER, 'checkpoints_article')
os.makedirs(CKPT_ROOT, exist_ok=True)

# Per-generator folders + results files
CKPT_FOLDERS  = {ds: os.path.join(CKPT_ROOT, ds) for ds in DS_LIST}
RESULTS_FILES = {ds: os.path.join(DRIVE_FOLDER, f'results_article_{ds}.json')
                 for ds in DS_LIST}
for f in CKPT_FOLDERS.values():
    os.makedirs(f, exist_ok=True)

# Training hyperparameters (IDENTICAL across all sessions)
COST_RATE      = 0.001
BATCH_SIZE     = 4096
CVAR_ALPHA     = 0.95
MSE_EPOCHS     = 500
MSE_LR         = 1e-3
MSE_PATIENCE   = 20
CVAR_EPOCHS    = 200
CVAR_LR        = 1e-4
CVAR_PATIENCE  = 20
SCHED_PATIENCE = 10
SCHED_FACTOR   = 0.5
SCHED_MIN_LR   = 1e-7
GRAD_CLIP      = 1.0

# Experimental design
OPTION_NAMES = ['basket_asian_call', 'asian_worst_of_put']
KAPPA_LEVELS = [0.95, 1.00, 1.05]
N_SEEDS      = 10

total_runs = len(DS_LIST) * len(OPTION_NAMES) * len(KAPPA_LEVELS) * N_SEEDS
print(f"\n  Session: {SESSION_NAME}")
print(f"  Generators: {DS_LIST}")
print(f"  Total runs: {total_runs} (each = MSE 500ep + CVaR 200ep)")

# ═════════════════════════════════════════════════════════════════
#  LOAD DATA
# ═════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print(f"LOADING DATA — {SESSION_NAME}")
print("=" * 70)

data_sources = {}
data_hashes  = {}

def sha256_array(arr):
    """Deterministic SHA256 of array/tensor (16 hex chars)."""
    if isinstance(arr, torch.Tensor):
        arr = arr.detach().cpu().numpy()
    return hashlib.sha256(np.ascontiguousarray(arr).tobytes()).hexdigest()[:16]

for ds in DS_LIST:
    fpath = os.path.join(DRIVE_FOLDER, f'{ds.lower()}_deep_hedging.npz')
    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f"Missing {fpath}. Run article_1_data.ipynb first.")
    _d = np.load(fpath)
    data_sources[ds] = {
        'S_train': torch.tensor(_d['S_train'], dtype=torch.float32, device=DEVICE),
        'S_val':   torch.tensor(_d['S_val'],   dtype=torch.float32, device=DEVICE),
        'S_test':  torch.tensor(_d['S_test'],  dtype=torch.float32, device=DEVICE),
    }
    data_hashes[ds] = {
        'S_train': sha256_array(_d['S_train']),
        'S_val':   sha256_array(_d['S_val']),
        'S_test':  sha256_array(_d['S_test']),
        'source_file': os.path.basename(fpath),
    }
    print(f"  {ds:>8s}: train {_d['S_train'].shape} "
          f"(SHA256:{data_hashes[ds]['S_train']})")
    del _d

# Shapes from first loaded generator
first_ds = DS_LIST[0]
_, T_plus_1, d = data_sources[first_ds]['S_train'].shape
T = T_plus_1 - 1
N_ASSETS = d

TICKERS = list(np.load(
    os.path.join(DRIVE_FOLDER, f'{first_ds.lower()}_deep_hedging.npz')
)['tickers'])

# ── Historical test paths (4 periods) ─────────────────────────────
hist_file = os.path.join(DRIVE_FOLDER, 'historical_test_paths.npz')
if not os.path.exists(hist_file):
    raise FileNotFoundError(f"Missing {hist_file}")

_h = np.load(hist_file)
hist_periods = {}
hist_hashes  = {}
for pname in _h['period_names']:
    ps = str(pname)
    hist_arr = _h[f'S_norm_{ps}']
    hist_periods[ps] = torch.tensor(hist_arr, dtype=torch.float32, device=DEVICE)
    hist_hashes[ps]  = sha256_array(hist_arr)
    print(f"  Hist {ps:<22s}: {hist_arr.shape} "
          f"(SHA256:{hist_hashes[ps]})")

data_hashes['historical'] = hist_hashes

# Sanity — must have 4 periods
expected_periods = {'COVID_2020', 'PostCOVID_2021_22',
                    'Normal_2023_24', 'Recent_2025'}
missing = expected_periods - set(hist_periods.keys())
if missing:
    raise RuntimeError(f"Missing historical periods: {missing}. "
                        f"Re-run article_1_data.ipynb.")

print(f"\n  Assets: {TICKERS}, d={N_ASSETS}, T={T}")
if DEVICE.type == 'cuda':
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"  GPU memory: {mem:.2f} GB used")


In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 2: MODEL COMPONENTS
#     HedgingNetwork, CVaRLoss, running_avg, payoffs,
#     forward engine, evaluate_full
# ═════════════════════════════════════════════════════════════════

def compute_running_averages(S_paths):
    """Causal running averages (no look-ahead)."""
    M, T1, d = S_paths.shape
    T = T1 - 1
    avg = torch.zeros(M, T, d, device=S_paths.device)
    avg[:, 0, :] = S_paths[:, 0, :]
    if T > 1:
        cumsum = S_paths[:, 1:, :].cumsum(dim=1)
        counts = torch.arange(1, T + 1, device=S_paths.device,
                              dtype=torch.float32).unsqueeze(0).unsqueeze(2)
        avg[:, 1:, :] = (cumsum / counts)[:, :-1, :]
    return avg


def payoff_basket_asian_call(S_paths, kappa=1.0):
    R_bar = S_paths[:, 1:, :].mean(dim=1)
    return torch.clamp(R_bar.mean(dim=1) - kappa, min=0.0)


def payoff_asian_worst_of_put(S_paths, kappa=1.0):
    R_bar = S_paths[:, 1:, :].mean(dim=1)
    return torch.clamp(kappa - R_bar.min(dim=1).values, min=0.0)


PAYOFF_FNS = {
    'basket_asian_call':   payoff_basket_asian_call,
    'asian_worst_of_put':  payoff_asian_worst_of_put,
}


class HedgingNetwork(nn.Module):
    """Feedforward hedging network — same across all experiments."""
    def __init__(self, d=3, hidden=(64, 64)):
        super().__init__()
        self.d = d
        self.hidden = tuple(hidden)
        layers = []
        prev = 3 * d + 1
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, d))
        self.net = nn.Sequential(*layers)
        nn.init.xavier_uniform_(self.net[-1].weight, gain=0.1)
        nn.init.zeros_(self.net[-1].bias)
        self.V0 = nn.Parameter(torch.tensor(0.0))

    def forward(self, spots, running_avg, delta_prev, time_left):
        x = torch.cat([spots, running_avg, delta_prev, time_left], dim=1)
        return self.net(x)

    @property
    def n_parameters(self):
        return sum(p.numel() for p in self.parameters())


class CVaRLoss(nn.Module):
    """Rockafellar-Uryasev CVaR loss with trainable threshold nu."""
    def __init__(self, alpha=0.95):
        super().__init__()
        self.alpha = alpha
        self.nu = nn.Parameter(torch.tensor(0.0))

    def forward(self, residuals):
        excess = torch.clamp(residuals - self.nu, min=0.0)
        return self.nu + excess.mean() / (1.0 - self.alpha)


def loss_mse(residuals):
    return (residuals ** 2).mean()


def deep_hedge_forward(net, S_paths, payoff_fn, kappa=1.0, cost_rate=0.001):
    """Forward pass through the hedging policy.
    Returns dict with residuals, payoff, pnl, cost, V0."""
    M, T1, d = S_paths.shape
    T = T1 - 1
    running_avg = compute_running_averages(S_paths)
    time_fracs = torch.arange(T, 0, -1, device=S_paths.device,
                              dtype=torch.float32) / T
    delta = torch.zeros(M, d, device=S_paths.device)
    pnl   = torch.zeros(M,    device=S_paths.device)
    cost  = torch.zeros(M,    device=S_paths.device)
    for t in range(T):
        delta_new = net(S_paths[:, t, :], running_avg[:, t, :],
                        delta, time_fracs[t].expand(M, 1))
        cost = cost + cost_rate * ((delta_new - delta).abs()
                                    * S_paths[:, t, :]).sum(dim=1)
        pnl  = pnl  + (delta_new * (S_paths[:, t+1, :]
                                     - S_paths[:, t, :])).sum(dim=1)
        delta = delta_new
    payoff = payoff_fn(S_paths, kappa)
    return {
        'residuals': payoff - net.V0 - pnl + cost,
        'payoff':    payoff.detach(),
        'pnl':       pnl.detach(),
        'cost':      cost.detach(),
        'V0':        net.V0.item(),
    }


@torch.no_grad()
def evaluate_full(net, S_paths, payoff_fn, kappa=1.0, cost_rate=0.001):
    """Full evaluation — returns metrics AND raw per-path arrays.

    Raw arrays are saved to checkpoint so reviewers can compute any
    distributional statistic without re-running the model.
    """
    net.eval()
    r = deep_hedge_forward(net, S_paths, payoff_fn, kappa, cost_rate)
    res = r['residuals']
    n = len(res)
    s = torch.sort(res).values
    metrics = {
        'rmse':     (res ** 2).mean().sqrt().item(),
        'mean':     res.mean().item(),
        'std':      res.std().item(),
        'cvar95':   s[int(np.ceil(0.95 * n)):].mean().item(),
        'cvar99':   s[int(np.ceil(0.99 * n)):].mean().item(),
        'max':      res.max().item(),
        'min':      res.min().item(),
        'V0':       r['V0'],
        'avg_cost': r['cost'].mean().item(),
        'avg_pnl':  r['pnl'].mean().item(),
        'n_paths':  int(n),
    }
    return {
        'metrics':   metrics,
        'residuals': res.detach().cpu().numpy().astype(np.float32),
        'pnl':       r['pnl'].cpu().numpy().astype(np.float32),
        'cost':      r['cost'].cpu().numpy().astype(np.float32),
        'payoff':    r['payoff'].cpu().numpy().astype(np.float32),
    }


print("  ✅ Model components defined")
print(f"     HedgingNetwork params: "
      f"{HedgingNetwork(d=N_ASSETS).n_parameters}")


In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 3: CHECKPOINTING + PROVENANCE + TRAINING LOOP
#     Q1-grade: 5-layer checkpoint, atomic saves, integrity hashes,
#     sidecar JSON, environment snapshot, RNG state capture
# ═════════════════════════════════════════════════════════════════

# ── Provenance helpers ────────────────────────────────────────────

def env_snapshot():
    """Freeze environment for reproducibility."""
    return {
        'python_version':  sys.version.split()[0],
        'platform':        platform.platform(),
        'torch_version':   torch.__version__,
        'numpy_version':   np.__version__,
        'cuda_available':  bool(torch.cuda.is_available()),
        'cuda_version':    (torch.version.cuda
                            if torch.cuda.is_available() else None),
        'cudnn_version':   (torch.backends.cudnn.version()
                            if torch.cuda.is_available() else None),
        'gpu_name':        (torch.cuda.get_device_name(0)
                            if torch.cuda.is_available() else None),
        'gpu_capability':  (list(torch.cuda.get_device_capability(0))
                            if torch.cuda.is_available() else None),
    }


def git_commit():
    """Current git commit hash, or None if not in a repo."""
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'],
            stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return None


def capture_rng_state():
    """Snapshot all RNG states for exact reproduction."""
    state = {
        'torch':  torch.get_rng_state().cpu().numpy().tolist(),
        'numpy':  [s.tolist() if hasattr(s, 'tolist') else s
                   for s in np.random.get_state()],
        'python': list(random.getstate()[1]) if random.getstate()[0] == 3
                  else None,
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = [s.cpu().numpy().tolist()
                               for s in torch.cuda.get_rng_state_all()]
    return state


def seed_all(seed):
    """Fully seed all RNGs for deterministic training."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ── Checkpoint save / load / verify ──────────────────────────────

NOTEBOOK_VERSION = 'article_v1.0'


def save_checkpoint(
    ckpt_path, net, cvar_loss, optimizer, scheduler,
    phase, config, history, best_val_loss,
    test_eval, hist_evals,
    data_hashes_dict, rng_state_snapshot,
    duration_seconds, mse_pretrained_epochs=None,
):
    """Save a full 5-layer checkpoint with integrity hash + JSON sidecar.

    Parameters
    ----------
    ckpt_path : str                — full path to .pt file
    net       : HedgingNetwork     — trained (best weights already loaded)
    cvar_loss : CVaRLoss | None
    optimizer : torch.optim.Adam
    scheduler : ReduceLROnPlateau
    phase     : 'mse' | 'cvar'
    config    : dict                — {ds, option, kappa, seed}
    history   : dict                — per-epoch training traces
    best_val_loss : float
    test_eval  : dict returned by evaluate_full (synthetic test)
    hist_evals : {period: dict from evaluate_full}
    data_hashes_dict : dict
    rng_state_snapshot : dict
    duration_seconds : float
    mse_pretrained_epochs : int | None — only for phase='cvar'
    """
    payload = {
        # ══ LAYER 1 — MODEL STATE ══════════════════════════════
        'net_state_dict':      net.state_dict(),
        'V0':                  net.V0.item(),
        'V0_requires_grad':    bool(net.V0.requires_grad),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'lr_final':            optimizer.param_groups[0]['lr'],

        # ══ LAYER 2 — ARCHITECTURE + HYPERPARAMETERS ═══════════
        'architecture': {
            'class_name':            'HedgingNetwork',
            'd':                     net.d,
            'hidden':                list(net.hidden),
            'activation':            'ReLU',
            'input_dim':             3 * net.d + 1,
            'input_features':        ['spots', 'running_avg',
                                      'delta_prev', 'time_left'],
            'output_dim':            net.d,
            'output_activation':     None,
            'init_method':           'xavier_uniform',
            'init_gain_last_layer':  0.1,
            'bias_init_last_layer':  0.0,
            'V0_init':               0.0,
            'n_parameters':          net.n_parameters,
        },
        'hyperparameters': {
            'COST_RATE':             COST_RATE,
            'BATCH_SIZE':            BATCH_SIZE,
            'CVAR_ALPHA':            CVAR_ALPHA,
            'MSE_EPOCHS':            MSE_EPOCHS,
            'MSE_LR':                MSE_LR,
            'MSE_PATIENCE':          MSE_PATIENCE,
            'CVAR_EPOCHS':           CVAR_EPOCHS,
            'CVAR_LR':               CVAR_LR,
            'CVAR_PATIENCE':         CVAR_PATIENCE,
            'scheduler_patience':    SCHED_PATIENCE,
            'scheduler_factor':      SCHED_FACTOR,
            'scheduler_min_lr':      SCHED_MIN_LR,
            'gradient_clip_norm':    GRAD_CLIP,
            'N_WINDOW':              T,
            'DELTA_T':               1.0 / 252.0,
            'M_SIMU':                20_000,
            'N_TRAIN':               16_000,
            'N_VAL':                 2_000,
            'N_TEST':                2_000,
            'curriculum':            (phase == 'cvar'),
            'V0_frozen_in_cvar':     (phase == 'cvar'),
            'mse_pretrained_epochs': mse_pretrained_epochs,
        },

        # ══ LAYER 3 — TRAINING HISTORY ═════════════════════════
        'phase':             phase,
        'n_epochs_trained':  len(history.get('train_loss', [])),
        'best_val_loss':     float(best_val_loss),
        'history':           history,

        # ══ LAYER 4 — EVALUATION ARTIFACTS (RAW ARRAYS) ════════
        'test_evaluation': {
            'residuals':      test_eval['residuals'],
            'pnl':            test_eval['pnl'],
            'cost':           test_eval['cost'],
            'payoff':         test_eval['payoff'],
            'metrics':        test_eval['metrics'],
            'residuals_hash': sha256_array(test_eval['residuals']),
            'M_paths':        int(len(test_eval['residuals'])),
        },
        'historical_evaluation': {
            period: {
                'residuals':      ev['residuals'],
                'pnl':            ev['pnl'],
                'cost':           ev['cost'],
                'payoff':         ev['payoff'],
                'metrics':        ev['metrics'],
                'residuals_hash': sha256_array(ev['residuals']),
                'M_paths':        int(len(ev['residuals'])),
            }
            for period, ev in hist_evals.items()
        },

        # ══ LAYER 5 — PROVENANCE ═══════════════════════════════
        'config':            config,
        'run_id':            f"{config['ds']}_{config['option']}_"
                             f"k{config['kappa']:.2f}_s{config['seed']}_{phase}",
        'timestamp_utc':     datetime.datetime.utcnow().isoformat() + 'Z',
        'duration_seconds':  float(duration_seconds),
        'environment':       env_snapshot(),
        'data_hashes':       data_hashes_dict,
        'notebook_version':  NOTEBOOK_VERSION,
        'git_commit':        git_commit(),
        'random_seed':       config['seed'],
        'rng_state':         rng_state_snapshot,
    }

    # CVaR-specific fields
    if cvar_loss is not None:
        payload['cvar_state_dict'] = cvar_loss.state_dict()
        payload['nu']    = float(cvar_loss.nu.item())
        payload['alpha'] = float(cvar_loss.alpha)

    # ── Integrity hash (SHA256 of entire payload minus the hash field) ─
    buf = io.BytesIO()
    torch.save(payload, buf)
    payload['checkpoint_hash'] = hashlib.sha256(buf.getvalue()).hexdigest()

    # ── Atomic save ────────────────────────────────────────────
    tmp_path = ckpt_path + '.tmp'
    torch.save(payload, tmp_path)
    os.replace(tmp_path, ckpt_path)

    # ── Sidecar JSON (human-readable metadata) ─────────────────
    sidecar = {
        'run_id':                 payload['run_id'],
        'phase':                  phase,
        'config':                 config,
        'n_epochs_trained':       payload['n_epochs_trained'],
        'best_val_loss':          payload['best_val_loss'],
        'lr_final':               payload['lr_final'],
        'V0':                     payload['V0'],
        'nu':                     payload.get('nu'),
        'test_metrics':           test_eval['metrics'],
        'historical_metrics':     {p: ev['metrics']
                                    for p, ev in hist_evals.items()},
        'test_residuals_hash':    payload['test_evaluation']['residuals_hash'],
        'historical_residuals_hashes': {
            p: payload['historical_evaluation'][p]['residuals_hash']
            for p in hist_evals.keys()
        },
        'timestamp_utc':          payload['timestamp_utc'],
        'duration_seconds':       payload['duration_seconds'],
        'environment':            payload['environment'],
        'data_hashes':            data_hashes_dict,
        'checkpoint_hash':        payload['checkpoint_hash'],
        'checkpoint_size_bytes':  os.path.getsize(ckpt_path),
        'architecture':           payload['architecture'],
        'hyperparameters':        payload['hyperparameters'],
        'git_commit':             payload['git_commit'],
        'notebook_version':       payload['notebook_version'],
    }
    sidecar_path = ckpt_path.replace('.pt', '.json')
    tmp_sidecar = sidecar_path + '.tmp'
    with open(tmp_sidecar, 'w') as f:
        json.dump(sidecar, f, indent=2, default=str)
    os.replace(tmp_sidecar, sidecar_path)

    return payload['checkpoint_hash']


def load_checkpoint(ckpt_path, DEVICE, strict_verify=True):
    """Reload a checkpoint; rebuild network from saved architecture spec."""
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if strict_verify:
        arch = ckpt['architecture']
        assert arch['class_name'] == 'HedgingNetwork', \
               f"Expected HedgingNetwork, got {arch['class_name']}"

    net = HedgingNetwork(d=ckpt['architecture']['d'],
                         hidden=tuple(ckpt['architecture']['hidden'])).to(DEVICE)
    net.load_state_dict(ckpt['net_state_dict'])
    net.eval()

    cvar_loss = None
    if 'cvar_state_dict' in ckpt:
        cvar_loss = CVaRLoss(alpha=ckpt['alpha']).to(DEVICE)
        cvar_loss.load_state_dict(ckpt['cvar_state_dict'])

    return net, cvar_loss, ckpt


def verify_checkpoint(ckpt_path, DEVICE='cpu'):
    """Full integrity check. Raises ValueError if corrupt."""
    required_top = [
        'net_state_dict', 'V0', 'architecture', 'hyperparameters',
        'phase', 'history', 'best_val_loss', 'n_epochs_trained',
        'test_evaluation', 'historical_evaluation',
        'config', 'run_id', 'timestamp_utc', 'environment',
        'data_hashes', 'random_seed', 'rng_state', 'checkpoint_hash',
        'optimizer_state_dict', 'scheduler_state_dict',
    ]

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    missing = [k for k in required_top if k not in ckpt]
    if missing:
        raise ValueError(f"{ckpt_path}: missing fields {missing}")

    # 4 historical periods
    expected_periods = {'COVID_2020', 'PostCOVID_2021_22',
                        'Normal_2023_24', 'Recent_2025'}
    actual = set(ckpt['historical_evaluation'].keys())
    mp = expected_periods - actual
    if mp:
        raise ValueError(f"{ckpt_path}: missing periods {mp}")

    for p, e in ckpt['historical_evaluation'].items():
        assert 'residuals' in e and e['residuals'].shape[0] > 0
        assert 'metrics' in e

    if ckpt['phase'] == 'cvar':
        for f in ['cvar_state_dict', 'nu', 'alpha']:
            if f not in ckpt:
                raise ValueError(f"CVaR checkpoint missing field {f}")

    # Verify integrity hash
    saved_hash = ckpt['checkpoint_hash']
    ckpt_no_hash = {k: v for k, v in ckpt.items() if k != 'checkpoint_hash'}
    buf = io.BytesIO()
    torch.save(ckpt_no_hash, buf)
    computed_hash = hashlib.sha256(buf.getvalue()).hexdigest()
    if saved_hash != computed_hash:
        raise ValueError(
            f"{ckpt_path}: integrity hash mismatch.\n"
            f"  Saved:    {saved_hash}\n"
            f"  Computed: {computed_hash}"
        )
    return True


# ── Training loop with gradient tracking ─────────────────────────

def train_loop(net, loss_fn, all_params, S_tr, S_vl, payoff_fn, kappa,
               lr, max_ep, patience, verbose_every=50):
    """Training loop; returns (best_state, n_epochs, wall_time,
    optimizer, scheduler, history, best_val_loss)."""
    M = S_tr.shape[0]
    nb = max(1, M // BATCH_SIZE)
    optimizer = optim.Adam(all_params, lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        patience=SCHED_PATIENCE,
        factor=SCHED_FACTOR,
        min_lr=SCHED_MIN_LR)

    best_val  = float('inf')
    pat_cnt   = 0
    best_state = None
    t0 = time.time()

    # Identify nu parameter (for CVaR phase) so we can track it per epoch
    nu_param = None
    for p in all_params:
        if p.numel() == 1 and p.requires_grad:
            # Heuristic: scalar trainable → either V0 or nu.
            # V0 frozen in CVaR phase, so this is nu.
            if p is not net.V0:
                nu_param = p
                break

    history = {
        'train_loss':          [],
        'val_loss':            [],
        'lr':                  [],
        'grad_norm_pre_clip':  [],
        'grad_norm_post_clip': [],
        'V0':                  [],
        'nu':                  [],
    }

    for epoch in range(max_ep):
        net.train()
        perm = torch.randperm(M, device=DEVICE)
        ep_loss    = 0.0
        gn_pre_sum  = 0.0
        gn_post_sum = 0.0

        for b in range(nb):
            idx = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            optimizer.zero_grad()
            res = deep_hedge_forward(net, S_tr[idx], payoff_fn,
                                     kappa, COST_RATE)['residuals']
            loss = loss_fn(res)
            loss.backward()

            # Grad norm before clip (infinite threshold → no clip)
            gn_pre = torch.nn.utils.clip_grad_norm_(all_params, float('inf'))
            # Actual clip at GRAD_CLIP=1.0
            gn_post = torch.nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)

            optimizer.step()
            ep_loss    += loss.item()
            gn_pre_sum  += (gn_pre.item()  if torch.is_tensor(gn_pre)
                            else float(gn_pre))
            gn_post_sum += (gn_post.item() if torch.is_tensor(gn_post)
                            else float(gn_post))

        ep_loss /= nb

        net.eval()
        with torch.no_grad():
            val_res = deep_hedge_forward(net, S_vl, payoff_fn,
                                         kappa, COST_RATE)['residuals']
            val_loss = loss_fn(val_res).item()
        scheduler.step(val_loss)

        history['train_loss'].append(float(ep_loss))
        history['val_loss'].append(float(val_loss))
        history['lr'].append(float(optimizer.param_groups[0]['lr']))
        history['grad_norm_pre_clip'].append(float(gn_pre_sum / nb))
        history['grad_norm_post_clip'].append(float(gn_post_sum / nb))
        history['V0'].append(float(net.V0.item()))
        history['nu'].append(float(nu_param.item()) if nu_param is not None
                             else None)

        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            pat_cnt    = 0
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        else:
            pat_cnt += 1

        if pat_cnt >= patience:
            break

        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"      ep {epoch+1:3d}  "
                  f"train={ep_loss:.6f}  val={val_loss:.6f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.1e}  "
                  f"|g|={gn_pre_sum/nb:.2f}")

    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    return (best_state, epoch + 1, time.time() - t0,
            optimizer, scheduler, history, float(best_val))


# ── Results-JSON I/O (atomic) ────────────────────────────────────

def load_results(ds):
    fp = RESULTS_FILES[ds]
    if os.path.exists(fp):
        with open(fp, 'r') as f:
            return json.load(f)
    return {}

def save_results(ds, results):
    fp = RESULTS_FILES[ds]
    tmp = fp + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    os.replace(tmp, fp)


def make_key(ds, opt, kappa, seed):
    return f"{ds}_article_{opt}_k{kappa:.2f}_s{seed}"


print("  ✅ Checkpointing + training helpers defined")


In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 4: MAIN RUN LOOP — curriculum MSE → CVaR per config
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print(f"RUNNING EXPERIMENTS — SESSION: {SESSION_NAME}")
print("=" * 70)

grand_start = time.time()

for ds in DS_LIST:
    ds_results = load_results(ds)
    ckpt_folder = CKPT_FOLDERS[ds]

    S_tr = data_sources[ds]['S_train']
    S_vl = data_sources[ds]['S_val']
    S_te = data_sources[ds]['S_test']
    ds_data_hashes = {
        'S_train':    data_hashes[ds]['S_train'],
        'S_val':      data_hashes[ds]['S_val'],
        'S_test':     data_hashes[ds]['S_test'],
        'source_file': data_hashes[ds]['source_file'],
        'historical': data_hashes['historical'],
    }

    # Build run queue (skip already-complete)
    run_queue = []
    for opt in OPTION_NAMES:
        for kappa in KAPPA_LEVELS:
            for seed in range(N_SEEDS):
                key = make_key(ds, opt, kappa, seed)
                if key not in ds_results:
                    run_queue.append((opt, kappa, seed, key))

    n_ds_total = len(OPTION_NAMES) * len(KAPPA_LEVELS) * N_SEEDS
    print(f"\n{'─' * 70}")
    print(f"  {ds}: {len(run_queue)} remaining / {n_ds_total} total")
    print(f"  Existing: {len(ds_results)} | Checkpoint folder: {ckpt_folder}")
    print(f"{'─' * 70}")

    if not run_queue:
        print(f"  ✅ {ds} already complete — skipping.")
        continue

    t_ds_start = time.time()

    for i, (opt, kappa, seed, key) in enumerate(run_queue):
        print(f"\n  ┌─ [{ds}] [{len(ds_results)+1}/{n_ds_total}] {key}")

        # ── Seed everything + snapshot RNG state ──
        seed_all(seed)
        rng_snapshot = capture_rng_state()

        payoff_fn = PAYOFF_FNS[opt]
        config = {'ds': ds, 'option': opt, 'kappa': kappa, 'seed': seed}

        # Instantiate fresh network (Xavier init with seed just set)
        net = HedgingNetwork(d=N_ASSETS, hidden=(64, 64)).to(DEVICE)

        # ═══ PHASE 1 — MSE (V0 trainable) ═══════════════════════
        t_p1 = time.time()
        (mse_state, mse_ep, mse_time,
         mse_opt, mse_sched, mse_hist, mse_best_val) = train_loop(
            net, loss_mse, list(net.parameters()), S_tr, S_vl,
            payoff_fn, kappa, MSE_LR, MSE_EPOCHS, MSE_PATIENCE,
            verbose_every=100)

        mse_test_eval  = evaluate_full(net, S_te, payoff_fn, kappa, COST_RATE)
        mse_hist_evals = {
            p: evaluate_full(net, S_h, payoff_fn, kappa, COST_RATE)
            for p, S_h in hist_periods.items()
        }

        mse_ckpt_path = os.path.join(ckpt_folder, f'{key}_mse.pt')
        mse_hash = save_checkpoint(
            ckpt_path=mse_ckpt_path, net=net, cvar_loss=None,
            optimizer=mse_opt, scheduler=mse_sched,
            phase='mse', config=config,
            history=mse_hist, best_val_loss=mse_best_val,
            test_eval=mse_test_eval, hist_evals=mse_hist_evals,
            data_hashes_dict=ds_data_hashes,
            rng_state_snapshot=rng_snapshot,
            duration_seconds=time.time() - t_p1,
        )
        verify_checkpoint(mse_ckpt_path)

        print(f"  │  MSE:  {mse_time:>5.0f}s ({mse_ep:>3d}ep)  "
              f"V₀={net.V0.item():+.4f}  "
              f"Std={mse_test_eval['metrics']['std']:.4f}  "
              f"CVaR95={mse_test_eval['metrics']['cvar95']:.4f}")

        # ═══ PHASE 2 — CVaR curriculum (V0 frozen) ══════════════
        t_p2 = time.time()
        net.load_state_dict(mse_state)            # start from best MSE
        net.V0.requires_grad = False              # freeze V0
        cvar_loss = CVaRLoss(alpha=CVAR_ALPHA).to(DEVICE)

        cvar_params = [p for p in net.parameters() if p.requires_grad] \
                    + list(cvar_loss.parameters())

        (cvar_state, cvar_ep, cvar_time,
         cvar_opt, cvar_sched, cvar_hist, cvar_best_val) = train_loop(
            net, cvar_loss, cvar_params, S_tr, S_vl,
            payoff_fn, kappa, CVAR_LR, CVAR_EPOCHS, CVAR_PATIENCE,
            verbose_every=50)

        cvar_test_eval  = evaluate_full(net, S_te, payoff_fn, kappa, COST_RATE)
        cvar_hist_evals = {
            p: evaluate_full(net, S_h, payoff_fn, kappa, COST_RATE)
            for p, S_h in hist_periods.items()
        }

        cvar_ckpt_path = os.path.join(ckpt_folder, f'{key}_cvar.pt')
        cvar_hash = save_checkpoint(
            ckpt_path=cvar_ckpt_path, net=net, cvar_loss=cvar_loss,
            optimizer=cvar_opt, scheduler=cvar_sched,
            phase='cvar', config=config,
            history=cvar_hist, best_val_loss=cvar_best_val,
            test_eval=cvar_test_eval, hist_evals=cvar_hist_evals,
            data_hashes_dict=ds_data_hashes,
            rng_state_snapshot=rng_snapshot,
            duration_seconds=time.time() - t_p2,
            mse_pretrained_epochs=mse_ep,
        )
        verify_checkpoint(cvar_ckpt_path)

        print(f"  │  CVaR: {cvar_time:>5.0f}s ({cvar_ep:>3d}ep)  "
              f"ν={cvar_loss.nu.item():+.4f}  "
              f"Std={cvar_test_eval['metrics']['std']:.4f}  "
              f"CVaR95={cvar_test_eval['metrics']['cvar95']:.4f}")

        # ═══ Append JSON entry (summary + checkpoint paths) ═════
        ds_results[key] = {
            'key':       key,
            'ds':        ds,
            'option':    opt,
            'kappa':     kappa,
            'seed':      seed,
            'mse': {
                'n_epochs':      mse_ep,
                'train_time_s':  round(mse_time, 1),
                'best_val_loss': mse_best_val,
                'V0':            float(net.V0.item()),
                'test':          mse_test_eval['metrics'],
                'historical':    {p: ev['metrics']
                                  for p, ev in mse_hist_evals.items()},
                'ckpt_path':     os.path.basename(mse_ckpt_path),
                'ckpt_hash':     mse_hash,
            },
            'cvar': {
                'n_epochs':      cvar_ep,
                'train_time_s':  round(cvar_time, 1),
                'best_val_loss': cvar_best_val,
                'V0':            float(net.V0.item()),
                'nu':            float(cvar_loss.nu.item()),
                'alpha':         CVAR_ALPHA,
                'test':          cvar_test_eval['metrics'],
                'historical':    {p: ev['metrics']
                                  for p, ev in cvar_hist_evals.items()},
                'ckpt_path':     os.path.basename(cvar_ckpt_path),
                'ckpt_hash':     cvar_hash,
                'mse_pretrained_epochs': mse_ep,
            },
            'completed_utc': datetime.datetime.utcnow().isoformat() + 'Z',
        }
        save_results(ds, ds_results)

        # ═══ Progress ═══════════════════════════════════════════
        elapsed = time.time() - t_ds_start
        eta = elapsed / (i + 1) * (len(run_queue) - i - 1)
        print(f"  └─ 💾 Saved — {i+1}/{len(run_queue)}  "
              f"ETA {eta/3600:.1f}h (this ds)")

        # ═══ Cleanup ════════════════════════════════════════════
        del net, cvar_loss, mse_opt, cvar_opt, mse_sched, cvar_sched
        del mse_test_eval, mse_hist_evals
        del cvar_test_eval, cvar_hist_evals
        clear_mem()

    print(f"\n  ✅ {ds} done: {len(ds_results)} runs "
          f"({(time.time()-t_ds_start)/3600:.1f}h)")

total_time = (time.time() - grand_start) / 3600
print(f"\n{'═' * 70}")
print(f"  SESSION {SESSION_NAME} COMPLETE")
print(f"  Runtime: {total_time:.1f}h")
print(f"{'═' * 70}")


In [ ]:
# ═════════════════════════════════════════════════════════════════
#  CELL 5: SESSION MANIFEST — aggregate all checkpoint hashes
# ═════════════════════════════════════════════════════════════════

print("=" * 70)
print(f"WRITING SESSION MANIFEST — {SESSION_NAME}")
print("=" * 70)

for ds in DS_LIST:
    ckpt_folder = CKPT_FOLDERS[ds]
    manifest = {
        'session':         SESSION_NAME,
        'generator':       ds,
        'notebook_version': NOTEBOOK_VERSION,
        'timestamp_utc':   datetime.datetime.utcnow().isoformat() + 'Z',
        'environment':     env_snapshot(),
        'git_commit':      git_commit(),
        'hyperparameters': {
            'COST_RATE':    COST_RATE,
            'BATCH_SIZE':   BATCH_SIZE,
            'CVAR_ALPHA':   CVAR_ALPHA,
            'MSE_EPOCHS':   MSE_EPOCHS,
            'MSE_LR':       MSE_LR,
            'CVAR_EPOCHS':  CVAR_EPOCHS,
            'CVAR_LR':      CVAR_LR,
            'N_SEEDS':      N_SEEDS,
            'seeds_used':   list(range(N_SEEDS)),
        },
        'data_hashes':     {
            'S_train':    data_hashes[ds]['S_train'],
            'S_val':      data_hashes[ds]['S_val'],
            'S_test':     data_hashes[ds]['S_test'],
            'source_file': data_hashes[ds]['source_file'],
            'historical': data_hashes['historical'],
        },
        'n_checkpoints':   0,
        'checkpoints':     [],
        'aggregated_hash': None,
    }

    ckpts = sorted([f for f in os.listdir(ckpt_folder) if f.endswith('.pt')])
    hashes = []
    total_size = 0
    corrupt = []

    print(f"\n  {ds}: verifying {len(ckpts)} checkpoints...")
    for ckpt_name in ckpts:
        ckpt_path = os.path.join(ckpt_folder, ckpt_name)
        sidecar_path = ckpt_path.replace('.pt', '.json')

        # Verify integrity
        try:
            verify_checkpoint(ckpt_path)
            status = 'OK'
        except Exception as e:
            status = f'CORRUPT: {e}'
            corrupt.append(ckpt_name)

        # Read sidecar for metadata
        if os.path.exists(sidecar_path):
            with open(sidecar_path) as f:
                sc = json.load(f)
            entry = {
                'file':             ckpt_name,
                'sidecar':          os.path.basename(sidecar_path),
                'run_id':           sc.get('run_id'),
                'phase':            sc.get('phase'),
                'config':           sc.get('config'),
                'n_epochs':         sc.get('n_epochs_trained'),
                'best_val_loss':    sc.get('best_val_loss'),
                'checkpoint_hash':  sc.get('checkpoint_hash'),
                'size_bytes':       sc.get('checkpoint_size_bytes'),
                'status':           status,
            }
            manifest['checkpoints'].append(entry)
            hashes.append(sc.get('checkpoint_hash', ''))
            total_size += sc.get('checkpoint_size_bytes', 0)
        else:
            print(f"    ⚠️  Missing sidecar for {ckpt_name}")

    manifest['n_checkpoints']    = len(ckpts)
    manifest['total_size_mb']    = round(total_size / 1e6, 2)
    manifest['n_corrupt']        = len(corrupt)
    manifest['corrupt_files']    = corrupt
    manifest['aggregated_hash']  = hashlib.sha256(
        ''.join(sorted(hashes)).encode()).hexdigest()

    manifest_path = os.path.join(DRIVE_FOLDER,
                                  f'manifest_article_{ds}.json')
    tmp = manifest_path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(manifest, f, indent=2, default=str)
    os.replace(tmp, manifest_path)

    print(f"  📋 {os.path.basename(manifest_path)}:  "
          f"{manifest['n_checkpoints']} ckpts  "
          f"({manifest['total_size_mb']} MB)")
    print(f"     Aggregated hash: {manifest['aggregated_hash'][:24]}...")
    if corrupt:
        print(f"     ⚠️  {len(corrupt)} corrupt file(s)!")

print(f"\n{'═' * 70}")
print(f"  ✅ SESSION {SESSION_NAME} — MANIFEST COMPLETE")
print(f"  Next: run article_5_merge.ipynb after all sessions finish")
print(f"{'═' * 70}")
